# AD vs FTD features: one row per subject, from the cleaned EEG

This notebook turns the cleaned recordings in `data/derivatives/` into a feature
table with **one row per subject**. `ADvsFTD_ds004504.ipynb` reads only that
table. Nothing here uses the diagnosis: labels are copied into `subjects.csv`
and never read.

### Why not reuse `subject_cache/`

The cache holds spectra of 20 s windows, but phase-lag connectivity, entropy and
fractal dimension need the time signal. The cache also contains two non-EEG
numbers that separate AD from FTD on their own:

| | AD | FTD | AD vs FTD AUC on its own |
|---|---|---|---|
| windows per subject in the cache (cap 80 / 100) | 79.4 | 99.7 | **1.000** |
| recording length | 13.5 min | 12.0 min | 0.669 |

No model is given these numbers. Still, they mean FTD features average over
minutes 7–8.6 of a recording, which no AD feature covers. So here **every
subject contributes the same 300 s**: the first 75 four-second epochs that
contain no ASR join.

### ASR joins

The dataset's preprocessing cut bad stretches out with ASR and joined the
pieces, leaving 512 `boundary` markers (group medians 2.5–4 per subject). A join
is a jump in the signal. Spectra barely notice it, which is why the cache could
ignore joins, but phase and fractal measures do, so no epoch here contains one.

Every subject has at least 75 clean epochs; the shortest recording has exactly
75. For half the subjects the 75th ends by 304 s. Join counts do not separate AD
from FTD (AUC 0.43).

### Output: `features/`

| file | contents |
|---|---|
| `eeg_features.csv` | 88 × 417: EEG features only, prefixed by group (`bp_` 95, `spec_` 172, `nl_` 80, `conn_` 70) |
| `subjects.csv` | diagnosis, age, sex, MMSE, recording length, joins, signal quality. **Never a model input**: labels and confounder checks only |
| `feature_dictionary.csv` | every column with its group, unit, definition and how epochs were combined |
| `wpli.npz` | the 19 × 19 wPLI matrix per subject and band, for figures |

Run from top to bottom. It takes about 30 s on 8 cores and needs nothing beyond
`requirements.txt`. Cell 6 stops the run if any feature fails its known-answer
test.

In [1]:
# ===========================================================================
# CELL 1 — IMPORTS AND CONFIGURATION
# ===========================================================================
import math, os, time, warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import signal
from scipy.sparse.csgraph import shortest_path

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

EXP_DIR   = Path.cwd()                      # experiments_ad_vs_ftd/
PROJECT   = EXP_DIR.parent
DATA_ROOT = PROJECT / "data"                # ds004504 with derivatives/
CACHE_DIR = PROJECT / "subject_cache"       # read only by the Cell 7 check
OUT_DIR   = EXP_DIR / "features"
assert (DATA_ROOT / "participants.tsv").exists(), \
    f"no dataset at {DATA_ROOT}: open this notebook from experiments_ad_vs_ftd/"
OUT_DIR.mkdir(exist_ok=True)

# ---- the data every subject contributes ------------------------------------
SFREQ    = 500.0
EPOCH_S  = 4.0               # non-overlapping epochs on a fixed 4 s grid from t = 0
N_EPOCHS = 75                # 75 x 4 s = 300 s; the shortest recording has exactly 75 clean epochs
EPOCH_N  = int(EPOCH_S * SFREQ)

CHANNELS = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2",
            "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]          # subject_cache order
BANDS = {"delta": (0.5, 4.0), "theta": (4.0, 8.0), "alpha": (8.0, 13.0),
         "beta": (13.0, 30.0), "gamma": (30.0, 45.0)}                       # subject_cache bands
FMIN, FMAX = 0.5, 45.0       # the derivatives are band-passed 0.5-45 Hz

REGIONS = {"frontal":   ["Fp1", "Fp2", "F3", "F4", "F7", "F8", "Fz"],
           "central":   ["C3", "C4", "Cz"],
           "temporal":  ["T3", "T4", "T5", "T6"],
           "parietal":  ["P3", "P4", "Pz"],
           "occipital": ["O1", "O2"]}
POSTERIOR = ["P3", "P4", "Pz", "O1", "O2"]
PAIRS = {"frontal":   [("Fp1", "Fp2"), ("F3", "F4"), ("F7", "F8")],         # (left, right)
         "central":   [("C3", "C4")],
         "temporal":  [("T3", "T4"), ("T5", "T6")],
         "parietal":  [("P3", "P4")],
         "occipital": [("O1", "O2")]}
RATIOS = [("theta", "alpha"), ("delta", "alpha"), ("theta", "beta"), ("alpha", "beta")]

# ---- feature parameters: fixed here, before any result is seen -------------
WELCH_SEG_S  = 2.0           # spectral: 2 s Hann segments, 50 % overlap -> 0.5 Hz bins
ALPHA_SEARCH = (6.0, 13.0)   # alpha frequency; starts at 6 Hz because dementia slows the rhythm
NL_DECIM     = 4             # nonlinear features at 125 Hz, so a one-sample lag is 8 ms
PE_ORDER     = 4             # permutation entropy: 4! = 24 patterns, 497 per epoch
SVD_ORDER    = 10            # SVD entropy: 10-sample (80 ms) embedding
HFD_KMAX     = 10            # Higuchi: lags of 8-80 ms
DFA_BANDS    = ["theta", "alpha"]
DFA_SCALES_S = (1.0, 20.0)   # envelope DFA; 20 s still leaves 15 windows in 300 s

GROUP_LABEL = {"C": 0, "A": 1, "F": 2}                                      # as in subject_cache
GROUP_NAME  = {"C": "HC", "A": "AD", "F": "FTD"}
N_JOBS = max(1, min(8, (os.cpu_count() or 2) - 2))
CH_IDX = {c: i for i, c in enumerate(CHANNELS)}


def idx(chs):
    return [CH_IDX[c] for c in chs]


print(f"data    : {DATA_ROOT}\noutput  : {OUT_DIR}\nworkers : {N_JOBS}")

data    : C:\Projects\Major Project\data
output  : C:\Projects\Major Project\experiments_ad_vs_ftd\features
workers : 8


In [2]:
# ===========================================================================
# CELL 2 — SUBJECTS, RECORDING LENGTH AND CLEAN EPOCHS
# ===========================================================================
def set_file(sid):
    path = DATA_ROOT / "derivatives" / sid / "eeg" / f"{sid}_task-eyesclosed_eeg.set"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def clean_epoch_starts(raw):
    """Start sample of every 4 s grid epoch with no ASR join ('boundary') inside, and the join count."""
    joins = np.array([a["onset"] for a in raw.annotations if a["description"] == "boundary"]) * SFREQ
    starts = np.arange(0, raw.n_times - EPOCH_N + 1, EPOCH_N)
    inside = (joins[None, :] > starts[:, None]) & (joins[None, :] < starts[:, None] + EPOCH_N)
    return starts[~inside.any(axis=1)], len(joins)


rows = []
for r in pd.read_csv(DATA_ROOT / "participants.tsv", sep="\t").itertuples(index=False):
    raw = mne.io.read_raw_eeglab(set_file(r.participant_id), preload=False)
    if raw.info["sfreq"] != SFREQ:
        raise ValueError(f"{r.participant_id}: {raw.info['sfreq']} Hz, expected {SFREQ:g}")
    missing = set(CHANNELS) - set(raw.ch_names)
    if missing:
        raise ValueError(f"{r.participant_id}: missing channels {sorted(missing)}")
    starts, n_joins = clean_epoch_starts(raw)
    rows.append(dict(participant_id=r.participant_id, group=GROUP_NAME[r.Group],
                     label=GROUP_LABEL[r.Group], sex=r.Gender, age=r.Age,
                     mmse=pd.to_numeric(str(r.MMSE).strip()),
                     duration_s=raw.n_times / SFREQ, n_joins=n_joins, n_clean_epochs=len(starts),
                     span_s=(starts[N_EPOCHS - 1] + EPOCH_N) / SFREQ if len(starts) >= N_EPOCHS else np.nan))
subjects = pd.DataFrame(rows)

short = subjects[subjects.n_clean_epochs < N_EPOCHS]
assert short.empty, f"fewer than {N_EPOCHS} clean epochs:\n{short}"

print(f"{len(subjects)} subjects: {subjects.group.value_counts().to_dict()}")
print("\nmedian per group  (span_s = where the 75th clean epoch ends)")
print(subjects.groupby("group")[["duration_s", "n_joins", "n_clean_epochs", "span_s"]].median().round(1))

88 subjects: {'AD': 36, 'HC': 29, 'FTD': 23}

median per group  (span_s = where the 75th clean epoch ends)
       duration_s  n_joins  n_clean_epochs  span_s
group                                             
AD          820.0      2.5           199.5   304.0
FTD         699.1      3.0           170.0   304.0
HC          814.6      4.0           199.0   304.0


---
## Spectral features (Cell 3)

Each 4 s epoch gets a Welch spectrum $P(f)$: 2 s Hann segments, 50 % overlap,
0.5 Hz bins. Each per-epoch value becomes one number per channel by taking the
**median over the 75 epochs**, so one bad epoch cannot move it. Channels are
then averaged within each region. Columns ending in `_iqr` hold the
interquartile range over epochs instead, which measures how stable the quantity
was over the 5 minutes.

| feature | definition | what it reflects |
|---|---|---|
| `bp_` (Experiment 1) | mean of $\log_{10}P(f)$ over the band's 0.2 Hz bins in 20 s windows, per channel: the cache's own definition | the existing baseline, recomputed on the same 300 s as everything else |
| absolute power | $\log_{10}\sum_{f\in b}P(f)\,\Delta f$ | amplitude, which also carries skull thickness and so partly sex |
| relative power | $P_b \,/\, P_{0.5\text{–}45}$ | spectral shape, independent of overall amplitude |
| band ratios | $\log_{10}$ of $\theta/\alpha$, $\delta/\alpha$, $\theta/\beta$, $\alpha/\beta$ | EEG slowing. A linear model on log powers can already form these, so they matter mainly for trees and for reading results |
| posterior / frontal | $\log_{10}\big(P_{\text{P3,P4,Pz,O1,O2}} \,/\, P_{\text{frontal}}\big)$ per band | the front-to-back gradient of each rhythm |
| asymmetry | $\log_{10}P_{\text{right}} - \log_{10}P_{\text{left}}$ over homologous pairs; `absasym` averages its absolute value over all 8 pairs | FTD often affects one hemisphere more than the other, on either side, so the size of the asymmetry is the hypothesis, not its sign |
| spectral entropy | $-\sum_f p_f \ln p_f / \ln N$ with $p_f = P(f)/\sum P$ (Inouye et al. 1991) | 1 = flat spectrum; low = power concentrated in one rhythm |
| alpha frequency | `iaf_cog`: centre of gravity $\sum f P(f) / \sum P(f)$ over 6–13 Hz of the posterior median spectrum (Klimesch 1999). `paf`: the highest peak above the 1/f background | slowing of the dominant rhythm |

Slowing, where power moves from α/β into δ/θ, is the classic AD finding. FTD is
often reported to have a near-normal EEG, which is why AD vs FTD is hard. It is
also why relative, regional and asymmetry measures are worth testing rather
than plain band power alone.

In [3]:
# ===========================================================================
# CELL 3 — LOADING, EXISTING BAND POWER, SPECTRAL FEATURES
# ===========================================================================
def load_subject(sid):
    """First N_EPOCHS clean epochs (N_EPOCHS, 19, 2000) and the continuous signal they
    come from (19, T), both in µV and in subject_cache channel order, plus the epoch starts."""
    with warnings.catch_warnings():                       # parallel workers do not inherit Cell 1's settings
        warnings.filterwarnings("ignore", message=".*boundary.*")   # the joins are handled below
        raw = mne.io.read_raw_eeglab(set_file(sid), preload=False, verbose="ERROR")
    starts = clean_epoch_starts(raw)[0][:N_EPOCHS]
    x = raw.get_data(picks=CHANNELS, start=0, stop=int(starts[-1]) + EPOCH_N) * 1e6
    return np.stack([x[:, s:s + EPOCH_N] for s in starts]), x, starts


def band_mask(freqs, band):
    lo, hi = BANDS[band]
    return (freqs >= lo) & ((freqs < hi) if hi < FMAX else (freqs <= hi))   # the bands tile 0.5-45 Hz


def summarise(out, name, a, iqr=False, with_global=False):
    """a: (epochs, 19). Median over epochs per channel, then the mean over each region's channels."""
    med = np.median(a, axis=0)
    spread = np.subtract(*np.percentile(a, [75, 25], axis=0))
    for region, chs in REGIONS.items():
        out[f"{name}_{region}"] = med[idx(chs)].mean()
        if iqr:
            out[f"{name}_{region}_iqr"] = spread[idx(chs)].mean()
    if with_global:
        out[f"{name}_global"] = med.mean()


# ---- Experiment 1: the band power the existing notebooks use ---------------
def legacy_bp(x20_volts):
    """subject_cache's `bp` for one 20 s block: (5, 19) mean log10 PSD over each band's
    0.2 Hz bins. The same psd_array_welch call as build_cache.ipynb (4 x 5 s, no overlap)."""
    out = []
    for lo, hi in BANDS.values():
        psd, _ = mne.time_frequency.psd_array_welch(
            x20_volts, SFREQ, fmin=lo, fmax=hi, n_fft=2500, n_per_seg=2500, n_overlap=0,
            average=None, verbose="ERROR")                             # (19, bins, 4 segments)
        out.append(np.log10(np.maximum(psd.mean(-1), 1e-30)).mean(-1))
    return np.stack(out)


def existing_band_power(ep):
    """The cache's 95 band powers, on the same 300 s as every other feature (15 x 20 s)."""
    x = np.concatenate(list(ep), axis=-1) * 1e-6                       # (19, 150000) in V, as the cache
    blocks = x.reshape(len(CHANNELS), -1, 10000).transpose(1, 0, 2)
    bp = np.mean([legacy_bp(b) for b in blocks], axis=0)
    return {f"bp_{b}_{c}": bp[k, i] for k, b in enumerate(BANDS) for i, c in enumerate(CHANNELS)}


# ---- engineered spectral features --------------------------------------------
def alpha_frequency(freqs, spec):
    """Centre of gravity of the spectrum in ALPHA_SEARCH, and the frequency of the highest
    peak that rises above the 1/f background there; NaN when nothing does.

    The background is a straight line in log-log fitted over 2-40 Hz outside the search
    range. On the raw spectrum a falling background drags a peak to lower frequencies."""
    m = (freqs >= ALPHA_SEARCH[0]) & (freqs <= ALPHA_SEARCH[1])
    f = freqs[m]
    cog = (f * spec[m]).sum() / spec[m].sum()
    fit = (freqs >= 2.0) & (freqs <= 40.0) & ~m
    slope, icpt = np.polyfit(np.log10(freqs[fit]), np.log10(spec[fit]), 1)
    r = np.log10(spec[m]) - (icpt + slope * np.log10(f))                 # power above the background
    peaks = np.flatnonzero((r[1:-1] > r[:-2]) & (r[1:-1] > r[2:]) & (r[1:-1] > 0)) + 1
    if peaks.size == 0:
        return cog, np.nan
    k = peaks[np.argmax(r[peaks])]
    y0, y1, y2 = r[k - 1:k + 2]
    return cog, f[k] + 0.5 * (y0 - y2) / (y0 - 2 * y1 + y2) * (f[1] - f[0])


def spectral_features(ep):
    nper = int(WELCH_SEG_S * SFREQ)
    freqs, psd = signal.welch(ep, fs=SFREQ, window="hann", nperseg=nper, noverlap=nper // 2, axis=-1)
    df = freqs[1] - freqs[0]
    power = np.stack([psd[..., band_mask(freqs, b)].sum(-1) * df for b in BANDS], -1)   # (E, 19, 5) µV²
    logp, total = np.log10(power), power.sum(-1)
    B = {b: k for k, b in enumerate(BANDS)}
    f = {}

    for b, k in B.items():
        summarise(f, f"spec_abspow_{b}", logp[..., k], iqr=True)
        summarise(f, f"spec_relpow_{b}", power[..., k] / total, iqr=True)
    for num, den in RATIOS:
        summarise(f, f"spec_ratio_{num}_{den}", logp[..., B[num]] - logp[..., B[den]], with_global=True)

    post, front = power[:, idx(POSTERIOR)].mean(1), power[:, idx(REGIONS["frontal"])].mean(1)   # (E, 5)
    pair_asym = {pair: np.median(logp[:, CH_IDX[pair[1]]] - logp[:, CH_IDX[pair[0]]], axis=0)  # (5,)
                 for pairs in PAIRS.values() for pair in pairs}
    for b, k in B.items():
        f[f"spec_postfront_{b}"] = np.median(np.log10(post[:, k] / front[:, k]))
        for region, pairs in PAIRS.items():
            f[f"spec_asym_{b}_{region}"] = np.mean([pair_asym[p][k] for p in pairs])
        f[f"spec_absasym_{b}"] = np.mean([abs(a[k]) for a in pair_asym.values()])

    keep = (freqs >= FMIN) & (freqs <= FMAX)
    p = psd[..., keep] / psd[..., keep].sum(-1, keepdims=True)
    summarise(f, "spec_sentropy", -(p * np.log(p)).sum(-1) / np.log(keep.sum()), iqr=True, with_global=True)

    posterior = np.median(psd[:, idx(POSTERIOR)], axis=0).mean(0)       # median over epochs, mean over channels
    f["spec_iaf_cog"], f["spec_paf"] = alpha_frequency(freqs, posterior)
    return f


print("spectral features ready")

spectral features ready


---
## Nonlinear features (Cell 4)

These are computed per 4 s epoch on the signal decimated to 125 Hz, so a
one-sample lag is 8 ms and every measure looks at time scales the 0.5–45 Hz
data actually contain. Then, as above, the median (and IQR) is taken over
epochs and the mean over each region.

| feature | definition | what it reflects |
|---|---|---|
| Hjorth activity | $\log_{10}\operatorname{var}(x)$ (Hjorth 1970) | total power: a duplicate of the spectral group, kept for completeness |
| Hjorth mobility | $\sqrt{\operatorname{var}(x')/\operatorname{var}(x)}\cdot f_s/2\pi$ | the signal's mean frequency in Hz; lower = slower EEG |
| Hjorth complexity | $\text{mobility}(x') \,/\, \text{mobility}(x)$ | how far the waveform is from a pure sine (= 1) |
| permutation entropy | Shannon entropy of the ordinal patterns of 4 consecutive samples, divided by $\ln 4!$ (Bandt & Pompe 2002) | irregularity of the ups and downs, ignoring amplitude |
| SVD entropy | entropy of the normalised singular values of the 10-sample delay embedding, divided by $\log_2 10$ (Roberts et al. 1999) | how many independent components the signal needs; low = a few dominant rhythms |
| Higuchi fractal dimension | slope of $\log L(k)$ against $\log(1/k)$ for $k = 1…10$ (Higuchi 1988) | roughness: 1 = smooth curve, 2 = white noise |
| zero-crossing rate | sign changes per second | for a Gaussian signal it equals 2 × mobility (Rice 1944); Cell 9 shows it is a near-copy here |
| DFA exponent | slope of $\log F(s)$ against $\log s$ for the θ and α **amplitude envelopes**, $s$ = 1–20 s (Peng et al. 1994; Linkenkaer-Hansen et al. 2001) | long-range temporal correlations: how a rhythm's strength persists over seconds |

DFA is applied to the envelope rather than the raw signal. On the raw signal,
DFA at sub-second scales mostly measures the spectral slope, which the
spectral group already holds. It runs on the whole 300 s, because a 4 s epoch
cannot hold a 20 s scale. Where an epoch was skipped for a join, the envelope
jumps once; that happens about once per subject (median).

In [4]:
# ===========================================================================
# CELL 4 — NONLINEAR FEATURES  (all vectorised over rows: one row = one channel-epoch)
# ===========================================================================
def hjorth(x, fs):
    """Hjorth (1970): activity as log10 variance, mobility in Hz, complexity. x: (N, n)."""
    v0, v1, v2 = x.var(-1), np.diff(x, axis=-1).var(-1), np.diff(x, n=2, axis=-1).var(-1)
    mobility = np.sqrt(v1 / v0)
    return np.log10(v0), mobility * fs / (2 * np.pi), np.sqrt(v2 / v1) / mobility


def perm_entropy(x, order=PE_ORDER):
    """Permutation entropy (Bandt & Pompe 2002), lag 1, divided by ln(order!). x: (N, n) -> (N,)."""
    n = x.shape[-1] - order + 1
    emb = np.stack([x[:, i:i + n] for i in range(order)], axis=-1)       # (N, n, order)
    code = (np.argsort(emb, axis=-1, kind="stable") * order ** np.arange(order)).sum(-1)
    code += np.arange(len(x))[:, None] * order ** order                  # one block of codes per row
    p = np.bincount(code.ravel(), minlength=len(x) * order ** order).reshape(len(x), -1) / n
    h = -(p * np.log(np.where(p > 0, p, 1.0))).sum(-1)
    return h / np.log(math.factorial(order))


def svd_entropy(x, order=SVD_ORDER):
    """SVD entropy (Roberts et al. 1999), lag 1, divided by log2(order). x: (N, n) -> (N,)."""
    n = x.shape[-1] - order + 1
    emb = np.stack([x[:, i:i + n] for i in range(order)], axis=1)        # (N, order, n)
    sv = np.sqrt(np.clip(np.linalg.eigvalsh(emb @ emb.transpose(0, 2, 1)), 0.0, None))
    p = sv / sv.sum(-1, keepdims=True)
    return -(p * np.log2(np.where(p > 0, p, 1.0))).sum(-1) / np.log2(order)


def higuchi_fd(x, kmax=HFD_KMAX):
    """Higuchi (1988) fractal dimension. x: (N, n) -> (N,); 1 = smooth curve, 2 = white noise."""
    N = x.shape[-1]
    ks = np.arange(1, kmax + 1)
    L = np.empty((len(x), kmax))
    for j, k in enumerate(ks):
        L[:, j] = np.mean([np.abs(np.diff(x[:, m::k], axis=-1)).sum(-1) * (N - 1)
                           / ((x[:, m::k].shape[-1] - 1) * k) / k for m in range(k)], axis=0)
    return np.polyfit(np.log(1.0 / ks), np.log(L).T, 1)[0]


def zero_crossing_rate(x, fs):
    """Sign changes per second of the mean-removed signal. x: (N, n) -> (N,)."""
    s = np.signbit(x - x.mean(-1, keepdims=True))
    return (s[:, 1:] != s[:, :-1]).sum(-1) / (x.shape[-1] / fs)


def dfa_exponent(x, fs, scales_s=DFA_SCALES_S, n_scales=10):
    """DFA exponent (Peng et al. 1994): linear detrending in non-overlapping windows. x: (C, n) -> (C,)."""
    y = np.cumsum(x - x.mean(-1, keepdims=True), axis=-1)
    scales = np.unique(np.geomspace(scales_s[0] * fs, scales_s[1] * fs, n_scales).astype(int))
    F = []
    for s in scales:
        seg = y[:, :y.shape[-1] // s * s].reshape(len(y), -1, s)
        t = np.arange(s) - (s - 1) / 2                                    # centred, so intercept = mean
        resid = seg - seg.mean(-1, keepdims=True) - (seg * t).sum(-1, keepdims=True) / (t ** 2).sum() * t
        F.append(np.sqrt((resid ** 2).mean(axis=(1, 2))))
    return np.polyfit(np.log(scales), np.log(np.array(F)), 1)[0]


def nonlinear_features(x, starts):
    """x: continuous (19, T) µV at 500 Hz; starts: the clean epochs' start samples."""
    fs, n = SFREQ / NL_DECIM, EPOCH_N // NL_DECIM
    x125 = signal.decimate(x, NL_DECIM, ftype="fir", zero_phase=True, axis=-1)   # anti-aliased
    st = starts // NL_DECIM
    ep = np.stack([x125[:, s:s + n] for s in st])                       # (E, 19, 500)
    flat = ep.reshape(-1, n)
    activity, mobility, complexity = hjorth(flat, fs)
    per_epoch = {"nl_hjorth_activity": activity, "nl_hjorth_mobility": mobility,
                 "nl_hjorth_complexity": complexity, "nl_perm_entropy": perm_entropy(flat),
                 "nl_svd_entropy": svd_entropy(flat), "nl_higuchi_fd": higuchi_fd(flat),
                 "nl_zero_crossings": zero_crossing_rate(flat, fs)}
    f = {}
    for name, v in per_epoch.items():
        summarise(f, name, v.reshape(len(ep), -1), iqr=True)

    for b in DFA_BANDS:                                                   # whole 300 s, not single epochs
        lo, hi = BANDS[b]
        env = np.abs(signal.hilbert(mne.filter.filter_data(x125, fs, lo, hi, verbose="ERROR"), axis=-1))
        exponent = dfa_exponent(np.concatenate([env[:, s:s + n] for s in st], axis=-1), fs)
        for region, chs in REGIONS.items():
            f[f"nl_dfa_{b}_{region}"] = exponent[idx(chs)].mean()
    return f


print("nonlinear features ready")

nonlinear features ready


---
## Connectivity features (Cell 5)

These are **phase-based functional connectivity** computed from the time
signal. The project's existing E1/E2 graphs are different: they measure how
similar two electrodes' *spectra* are.

Each 4 s epoch gets Hann-tapered Fourier coefficients $X_i(f)$ in 0.25 Hz bins.
The cross-spectrum is $S_{ij}(f) = X_i(f)\,X_j^*(f)$, and $E[\cdot]$ averages
over the 75 epochs. Both measures are computed per frequency, then averaged over
the band.

| feature | definition | what it reflects |
|---|---|---|
| wPLI | $\lvert E[\operatorname{Im} S_{ij}]\rvert \,/\, E[\lvert\operatorname{Im} S_{ij}\rvert]$ (Vinck et al. 2011) | how consistently one electrode leads or lags another |
| imaginary coherency | $\big\lvert\operatorname{Im}\big(E[S_{ij}] / \sqrt{E[S_{ii}]\,E[S_{jj}]}\big)\big\rvert$ (Nolte et al. 2004) | the same idea, weighted by amplitude |
| mean | mean over the 171 electrode pairs | overall coupling in the band |
| regional strength | each electrode's mean coupling to the other 18, averaged over the region | where coupling is lost or kept |
| global efficiency | mean of $1/d_{ij}$, with $d_{ij}$ the shortest path when an edge's length is $1/w$ (Latora & Marchiori 2001) | how directly every electrode reaches every other |
| clustering | $\frac{1}{k(k-1)}\sum_{j,h}(\hat w_{ij}\hat w_{jh}\hat w_{hi})^{1/3}$ with $\hat w = w/\max w$ (Onnela et al. 2005) | how often an electrode's partners are coupled to each other |

**Why only lagged coupling.** A single source seen by two electrodes through
the scalp reaches both at the same moment. Both measures ignore zero-lag
coupling (Cell 6 checks this), so volume conduction cannot inflate them. The
price is that genuine zero-lag coupling is missed too.

**The noise floor.** wPLI is biased upward by an amount set by the number of
epochs: with 75, unrelated noise already scores about 0.13 (Cell 9). Every
subject has exactly 75 epochs, so the bias is the same for everyone and cannot
separate the classes. It does mean the signal is small.

In [5]:
# ===========================================================================
# CELL 5 — CONNECTIVITY FEATURES  (phase lag, from the time-domain signal)
# ===========================================================================
def fourier(ep):
    """Hann-tapered Fourier coefficients of each epoch in 0.5-45 Hz: freqs (F,), X (E, C, F)."""
    X = np.fft.rfft(signal.detrend(ep, axis=-1) * signal.windows.hann(ep.shape[-1]), axis=-1)
    freqs = np.fft.rfftfreq(ep.shape[-1], 1 / SFREQ)
    keep = (freqs >= FMIN) & (freqs <= FMAX)
    return freqs[keep], X[..., keep]


def phase_lag_matrices(freqs, X):
    """Per band: wPLI (Vinck et al. 2011) and |imaginary coherency| (Nolte et al. 2004),
    each (C, C), computed per frequency over epochs and then averaged over the band."""
    wpli, icoh = {}, {}
    for b in BANDS:
        Xb = X[..., band_mask(freqs, b)]
        S = Xb[:, :, None, :] * Xb[:, None, :, :].conj()               # cross-spectra (E, C, C, F_b)
        num, den = np.abs(S.imag.sum(0)), np.abs(S.imag).sum(0)
        w = np.divide(num, den, out=np.zeros_like(num), where=den > 0).mean(-1)
        Sm = S.mean(0)
        auto = np.real(np.einsum("iif->if", Sm))
        c = np.abs(Sm.imag / np.sqrt(auto[:, None] * auto[None, :])).mean(-1)
        np.fill_diagonal(w, 0.0)
        np.fill_diagonal(c, 0.0)
        wpli[b], icoh[b] = w, c
    return wpli, icoh


def global_efficiency(W):
    """Weighted global efficiency (Latora & Marchiori 2001): mean of 1 / shortest path, length = 1 / weight."""
    D = np.divide(1.0, W, out=np.full_like(W, np.inf), where=W > 0)
    np.fill_diagonal(D, 0.0)
    L = shortest_path(D, method="FW", directed=False)
    return (1.0 / L[~np.eye(len(W), dtype=bool)]).mean()


def mean_clustering(W):
    """Weighted clustering coefficient (Onnela et al. 2005), averaged over nodes."""
    c = np.cbrt(W / W.max())
    np.fill_diagonal(c, 0.0)
    k = (W > 0).sum(1)
    tri = np.diag(c @ c @ c)
    return np.divide(tri, k * (k - 1), out=np.zeros_like(tri), where=k > 1).mean()


def connectivity_features(ep):
    """Region and graph summaries of the band wPLI / imaginary-coherency matrices, plus wPLI (5, 19, 19)."""
    wpli, icoh = phase_lag_matrices(*fourier(ep))
    off = ~np.eye(len(CHANNELS), dtype=bool)
    f = {}
    for b in BANDS:
        for name, M in (("wpli", wpli[b]), ("icoh", icoh[b])):
            f[f"conn_{name}_mean_{b}"] = M[off].mean()
            strength = M.sum(1) / (len(CHANNELS) - 1)                    # each electrode's mean coupling
            for region, chs in REGIONS.items():
                f[f"conn_{name}_strength_{b}_{region}"] = strength[idx(chs)].mean()
        f[f"conn_wpli_efficiency_{b}"] = global_efficiency(wpli[b])
        f[f"conn_wpli_clustering_{b}"] = mean_clustering(wpli[b])
    return f, np.stack([wpli[b] for b in BANDS]).astype(np.float32)


print("connectivity features ready")

connectivity features ready


---
## Self-tests (Cell 6)

Every feature runs on a signal whose answer is known from theory:
- A pure sine has Hjorth complexity 1, two singular values and fractal dimension 1.
- White noise has permutation entropy 1, fractal dimension 2 and DFA exponent 0.5.
- A random walk has DFA exponent 1.5.
- Doubling the right hemisphere's amplitude gives an asymmetry of $\log_{10}4$.
- A 24 ms lagged copy of a 10 Hz source has wPLI near 1, and a zero-lag copy near 0.

The cell stops the notebook if any check fails, so a table is never built from
a broken formula.

In [6]:
# ===========================================================================
# CELL 6 — SELF-TESTS: every feature on a signal whose answer is known
# ===========================================================================
rng = np.random.default_rng(0)
fs125 = SFREQ / NL_DECIM
t = np.arange(EPOCH_N // NL_DECIM) / fs125                              # one 4 s epoch at 125 Hz
white = rng.standard_normal((200, len(t)))
sine10 = np.sin(2 * np.pi * 10 * t)[None]
checks = []


def expect(what, value, lo, hi):
    target = f"{lo:g} to {hi:g}" if hi - lo > 1e-6 else f"{(lo + hi) / 2:.3f}"
    checks.append((what, float(value), target, bool(lo <= value <= hi)))


# ---- nonlinear ---------------------------------------------------------------
_, mob, comp = hjorth(sine10, fs125)
expect("Hjorth mobility, 10 Hz sine (Hz)", mob[0], 9.8, 10.2)
expect("Hjorth complexity, sine", comp[0], 0.99, 1.01)
expect("zero crossings, 10 Hz sine (per s)", zero_crossing_rate(sine10, fs125)[0], 19.5, 20.5)
expect("permutation entropy, white noise", perm_entropy(white).mean(), 0.98, 1.0)
expect("permutation entropy, rising ramp", perm_entropy(t[None])[0], 0.0, 1e-9)
expect("SVD entropy, white noise", svd_entropy(white).mean(), 0.95, 1.0)
expect("SVD entropy, sine (2 singular values)", svd_entropy(sine10)[0], 0.25, 0.31)
expect("Higuchi FD, white noise", higuchi_fd(white).mean(), 1.9, 2.05)
expect("Higuchi FD, 1 Hz sine", higuchi_fd(np.sin(2 * np.pi * t)[None])[0], 0.95, 1.05)
noise300 = rng.standard_normal((20, int(300 * fs125)))
expect("DFA exponent, white noise", dfa_exponent(noise300, fs125).mean(), 0.45, 0.55)
expect("DFA exponent, random walk", dfa_exponent(np.cumsum(noise300, -1), fs125).mean(), 1.4, 1.6)

# ---- spectral ----------------------------------------------------------------
f_grid = np.fft.rfftfreq(int(WELCH_SEG_S * SFREQ), 1 / SFREQ)
tiles = sum(band_mask(f_grid, b).astype(int) for b in BANDS)
expect("bands cover 0.5-45 Hz exactly once", np.array_equal(tiles, (f_grid >= FMIN) & (f_grid <= FMAX)), 1, 1)
t500 = np.arange(EPOCH_N) / SFREQ
phase = rng.uniform(0, 2 * np.pi, (N_EPOCHS, len(CHANNELS), 1))
alpha_eeg = spectral_features(20 * np.sin(2 * np.pi * 10 * t500 + phase)
                              + rng.standard_normal((N_EPOCHS, len(CHANNELS), EPOCH_N)))
expect("alpha centre of gravity, 10 Hz rhythm", alpha_eeg["spec_iaf_cog"], 9.95, 10.05)
expect("alpha peak frequency, 10 Hz rhythm", alpha_eeg["spec_paf"], 9.95, 10.05)
expect("relative alpha power, 10 Hz rhythm", alpha_eeg["spec_relpow_alpha_occipital"], 0.98, 1.0)
f_half = np.arange(0, 45.5, 0.5)                                         # weak 9 Hz peak; 6 Hz edge is higher
steep = 1 / np.maximum(f_half, 0.5) ** 2 + 0.004 * np.exp(-((f_half - 9.0) / 0.8) ** 2)
expect("alpha peak frequency, weak 9 Hz peak on 1/f", alpha_frequency(f_half, steep)[1], 8.8, 9.1)
gain = np.ones((len(CHANNELS), 1))
gain[idx([r for pairs in PAIRS.values() for _, r in pairs])] = 2.0       # right side twice the amplitude
lopsided = spectral_features(gain * rng.standard_normal((N_EPOCHS, len(CHANNELS), EPOCH_N)))
expect("spectral entropy, white noise", lopsided["spec_sentropy_global"], 0.95, 1.0)
expect("asymmetry, right x2 amplitude (log10 4 = 0.602)", lopsided["spec_asym_alpha_temporal"], 0.55, 0.65)
expect("|asymmetry|, same", lopsided["spec_absasym_alpha"], 0.55, 0.65)

# ---- connectivity: a 10 Hz source seen with a lag, with no lag, and not at all ----
src = mne.filter.filter_data(rng.standard_normal((N_EPOCHS, 1, EPOCH_N + 2000)), SFREQ, 8.0, 13.0, verbose="ERROR")
lag = 12                                                                 # 24 ms, about a quarter cycle at 10 Hz


def noisy(s):
    return s + 0.05 * rng.standard_normal((N_EPOCHS, 1, EPOCH_N))


toy = np.concatenate([noisy(src[..., 1000:1000 + EPOCH_N]),
                      noisy(src[..., 1000 - lag:1000 - lag + EPOCH_N]),
                      noisy(src[..., 1000:1000 + EPOCH_N]),
                      noisy(np.zeros((N_EPOCHS, 1, EPOCH_N)))], axis=1)
W, C = phase_lag_matrices(*fourier(toy))
expect("wPLI alpha, 24 ms lagged copy", W["alpha"][0, 1], 0.9, 1.0)
expect("wPLI alpha, zero-lag copy (volume conduction)", W["alpha"][0, 2], 0.0, 0.3)
expect("wPLI alpha, unrelated noise", W["alpha"][0, 3], 0.0, 0.3)
expect("imaginary coherency alpha, lagged copy", C["alpha"][0, 1], 0.5, 1.0)
expect("imaginary coherency alpha, zero-lag copy", C["alpha"][0, 2], 0.0, 0.1)
U = np.full((19, 19), 0.5)
np.fill_diagonal(U, 0.0)
expect("efficiency, uniform graph w = 0.5", global_efficiency(U), 0.5 - 1e-9, 0.5 + 1e-9)
expect("clustering, uniform graph", mean_clustering(U), 1 - 1e-9, 1 + 1e-9)
chain = np.array([[0, 1, 0.1], [1, 0, 1], [0.1, 1, 0]])                 # a->c is shorter via b
expect("efficiency uses the shortest path (5/6)", global_efficiency(chain), 5 / 6 - 1e-9, 5 / 6 + 1e-9)

report = pd.DataFrame(checks, columns=["check", "value", "expected", "pass"])
print(report.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
assert report["pass"].all(), "a feature failed its self-test: fix it before building the table"
print(f"\nall {len(report)} self-tests pass")

                                          check  value      expected  pass
               Hjorth mobility, 10 Hz sine (Hz)  9.886   9.8 to 10.2  True
                        Hjorth complexity, sine  1.003  0.99 to 1.01  True
             zero crossings, 10 Hz sine (per s) 19.750  19.5 to 20.5  True
               permutation entropy, white noise  0.994     0.98 to 1  True
               permutation entropy, rising ramp -0.000         0.000  True
                       SVD entropy, white noise  0.999     0.95 to 1  True
          SVD entropy, sine (2 singular values)  0.299  0.25 to 0.31  True
                        Higuchi FD, white noise  2.000   1.9 to 2.05  True
                          Higuchi FD, 1 Hz sine  1.009  0.95 to 1.05  True
                      DFA exponent, white noise  0.496  0.45 to 0.55  True
                      DFA exponent, random walk  1.497    1.4 to 1.6  True
             bands cover 0.5-45 Hz exactly once  1.000         1.000  True
          alpha centre of

---
## One subject first (Cell 7)

This cell checks that the `bp_` columns are the cache's band power: on the same
20 s, they must reproduce sub-001's first cached window. It then times one
subject, so you know how long Cell 8 will take.

In [7]:
# ===========================================================================
# CELL 7 — ONE SUBJECT FIRST: MATCH WITH subject_cache, AND TIMING
# ===========================================================================
def subject_features(sid):
    """Every feature of one subject: (EEG features, signal-quality numbers, wPLI (5, 19, 19))."""
    ep, x, starts = load_subject(sid)
    feats = {"participant_id": sid, **existing_band_power(ep), **spectral_features(ep),
             **nonlinear_features(x, starts)}
    conn, wpli = connectivity_features(ep)
    feats.update(conn)
    quality = {"participant_id": sid,
               "q_rms_uv": float(np.median(np.sqrt((ep ** 2).mean(-1)))),       # typical amplitude
               "q_ptp99_uv": float(np.percentile(np.ptp(ep, axis=-1), 99))}    # largest swings
    return feats, quality, wpli


# The bp_ columns must be the cache's band power: on the same 20 s they have to
# reproduce sub-001's first cached window.
if (CACHE_DIR / "sub-001.npz").exists():
    raw = mne.io.read_raw_eeglab(set_file("sub-001"), preload=False)
    with np.load(CACHE_DIR / "sub-001.npz") as z:
        diff = np.abs(legacy_bp(raw.get_data(picks=CHANNELS, start=0, stop=10000)) - z["bp"][0]).max()
    print(f"existing band power vs subject_cache (sub-001, window 1): max |diff| {diff:.1e}")
    assert diff < 1e-4, "the bp_ columns would not be the cache's band power"
else:
    print("subject_cache/ not found: band-power match skipped")

t0 = time.time()
f1, q1, w1 = subject_features("sub-001")
secs = time.time() - t0
counts = pd.Series([c.split("_")[0] for c in f1 if c != "participant_id"]).value_counts()
print(f"sub-001: {len(f1) - 1} features in {secs:.1f} s  {counts.to_dict()}")
print(f"all {len(subjects)} subjects: about {secs * np.ceil(len(subjects) / N_JOBS) / 60:.1f} min on {N_JOBS} workers")

existing band power vs subject_cache (sub-001, window 1): max |diff| 4.6e-07


sub-001: 417 features in 1.0 s  {'spec': 172, 'bp': 95, 'nl': 80, 'conn': 70}
all 88 subjects: about 0.2 min on 8 workers


---
## Build (Cell 8)

All 88 subjects are built in parallel; HC subjects are included for later
AD-vs-HC and 3-class experiments. Nothing is written unless every subject
succeeds and every value except `spec_paf` is finite. Rerunning overwrites
`features/` with identical numbers, because no step is random.

In [8]:
# ===========================================================================
# CELL 8 — BUILD ALL SUBJECTS AND WRITE features/
# ===========================================================================
STEMS = {   # column stem -> (unit, definition, how the epochs are combined)
    "bp":                   ("log10 V²/Hz", "existing band power (Experiment 1): mean log10 PSD over the band's 0.2 Hz bins, the subject_cache definition", "mean over 15 windows of 20 s"),
    "spec_abspow":          ("log10 µV²", "absolute band power", "epochs"),
    "spec_relpow":          ("fraction", "band power / power in 0.5-45 Hz", "epochs"),
    "spec_ratio":           ("log10 ratio", "power ratio of the two named bands", "epochs"),
    "spec_postfront":       ("log10 ratio", "power over P3 P4 Pz O1 O2 / power over the frontal channels", "median over epochs"),
    "spec_asym":            ("log10 ratio", "right / left power over the region's homologous pairs (> 0: right larger)", "median over epochs"),
    "spec_absasym":         ("log10 ratio", "mean |right / left| over all 8 homologous pairs: asymmetry on either side", "median over epochs"),
    "spec_sentropy":        ("0-1", "normalised Shannon entropy of the 0.5-45 Hz spectrum (1 = flat)", "epochs"),
    "spec_iaf_cog":         ("Hz", "centre of gravity of the posterior spectrum in 6-13 Hz", "median spectrum over epochs"),
    "spec_paf":             ("Hz", "frequency of the highest posterior peak above the 1/f background in 6-13 Hz; empty when there is none", "median spectrum over epochs"),
    "nl_hjorth_activity":   ("log10 µV²", "Hjorth activity: variance of the signal", "epochs"),
    "nl_hjorth_mobility":   ("Hz", "Hjorth mobility sqrt(var x' / var x) in Hz: the mean frequency", "epochs"),
    "nl_hjorth_complexity": ("ratio", "Hjorth complexity mobility(x') / mobility(x); 1 = pure sine", "epochs"),
    "nl_perm_entropy":      ("0-1", "permutation entropy, order 4, 8 ms lag", "epochs"),
    "nl_svd_entropy":       ("0-1", "SVD entropy, 10-sample (80 ms) embedding", "epochs"),
    "nl_higuchi_fd":        ("1-2", "Higuchi fractal dimension, k = 1-10 (8-80 ms)", "epochs"),
    "nl_zero_crossings":    ("1/s", "zero crossings per second", "epochs"),
    "nl_dfa":               ("exponent", "DFA exponent of the band's amplitude envelope over 1-20 s", "the whole 300 s"),
    "conn_wpli_mean":       ("0-1", "mean wPLI over the 171 electrode pairs", "75 epochs"),
    "conn_wpli_strength":   ("0-1", "each electrode's mean wPLI to the 18 others, averaged over the region", "75 epochs"),
    "conn_wpli_efficiency": ("0-1", "weighted global efficiency of the wPLI graph", "75 epochs"),
    "conn_wpli_clustering": ("0-1", "weighted clustering coefficient of the wPLI graph", "75 epochs"),
    "conn_icoh_mean":       ("0-1", "mean |imaginary coherency| over the 171 electrode pairs", "75 epochs"),
    "conn_icoh_strength":   ("0-1", "each electrode's mean |imaginary coherency|, averaged over the region", "75 epochs"),
}
GROUPS = {"bp": "existing band power", "spec": "spectral", "nl": "nonlinear", "conn": "connectivity"}


def feature_dictionary(columns):
    rows = []
    for col in columns:
        stems = [s for s in STEMS if col == s or col.startswith(s + "_")]
        if not stems:
            raise KeyError(f"no definition for column {col}")
        unit, text, agg = STEMS[max(stems, key=len)]
        if agg == "epochs":
            agg = ("IQR" if col.endswith("_iqr") else "median") + " over 75 epochs, mean over the region's channels"
        rows.append(dict(column=col, group=GROUPS[col.split("_")[0]], unit=unit, definition=text, aggregation=agg))
    return pd.DataFrame(rows)


def safe(sid):
    try:
        return sid, subject_features(sid), None
    except Exception as exc:                                              # noqa: BLE001
        return sid, None, repr(exc)


t0 = time.time()
done = Parallel(n_jobs=N_JOBS, backend="loky")(delayed(safe)(sid) for sid in subjects.participant_id)
errors = [(sid, err) for sid, _, err in done if err]
for sid, err in errors:
    print(f"  {sid}: {err}")
assert not errors, f"{len(errors)} subjects failed; nothing was written"

eeg = pd.DataFrame([r[0] for _, r, _ in done]).set_index("participant_id")
subj = subjects.set_index("participant_id").join(pd.DataFrame([r[1] for _, r, _ in done]).set_index("participant_id"))
wpli = np.stack([r[2] for _, r, _ in done])
assert list(eeg.index) == list(subj.index)
finite = np.isfinite(eeg.drop(columns="spec_paf")).all()
assert finite.all(), f"non-finite values in {list(finite.index[~finite])}"

eeg.to_csv(OUT_DIR / "eeg_features.csv", float_format="%.6g")
subj.to_csv(OUT_DIR / "subjects.csv", float_format="%.6g")
feature_dictionary(eeg.columns).to_csv(OUT_DIR / "feature_dictionary.csv", index=False)
np.savez_compressed(OUT_DIR / "wpli.npz", wpli=wpli, participant_id=np.array(eeg.index),
                    bands=np.array(list(BANDS)), channels=np.array(CHANNELS))
print(f"{eeg.shape[0]} subjects x {eeg.shape[1]} EEG features in {time.time() - t0:.0f} s")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name:24s} {p.stat().st_size / 1024:7.0f} KB")

88 subjects x 417 EEG features in 33 s
  eeg_features.csv             337 KB
  feature_dictionary.csv        60 KB
  subjects.csv                   5 KB
  wpli.npz                     396 KB


---
## What is in the table (Cell 9)

This cell reads `features/` back and checks the features against each other,
not against the diagnosis.

In [9]:
# ===========================================================================
# CELL 9 — WHAT IS IN THE TABLE  (reads features/ only; no diagnosis is used)
# ===========================================================================
eeg = pd.read_csv(OUT_DIR / "eeg_features.csv", index_col=0)
groups = pd.read_csv(OUT_DIR / "feature_dictionary.csv").groupby("group", sort=False).size()
print(f"{eeg.shape[0]} subjects x {eeg.shape[1]} EEG features:  {groups.to_dict()}")
n_nan = eeg.isna().sum()
print("missing values:", n_nan[n_nan > 0].to_dict() or "none")
print(f"alpha centre of gravity: median {eeg.spec_iaf_cog.median():.2f} Hz, "
      f"range {eeg.spec_iaf_cog.min():.2f}-{eeg.spec_iaf_cog.max():.2f} Hz")

# Theory says some features are near-copies of others. If these numbers are far
# from the prediction, a formula is wrong; if they match, the copies add nothing.
zc = eeg.filter(regex=r"^nl_zero_crossings_[a-z]+$").to_numpy().ravel()
mob = eeg.filter(regex=r"^nl_hjorth_mobility_[a-z]+$").to_numpy().ravel()
print(f"\nzero crossings / (2 x Hjorth mobility): median {np.median(zc / (2 * mob)):.2f}, "
      f"r = {np.corrcoef(zc, mob)[0, 1]:.3f}   (Rice: 1 and 1 for a Gaussian signal)")
# wPLI is biased upward by an amount set by the number of epochs: measure it on
# 19 unrelated noise channels with the same 75 epochs.
floor_w, _ = phase_lag_matrices(*fourier(np.random.default_rng(1).standard_normal((N_EPOCHS, len(CHANNELS), EPOCH_N))))
off = ~np.eye(len(CHANNELS), dtype=bool)
rows = {b: dict(efficiency=np.corrcoef(eeg[f"conn_wpli_efficiency_{b}"], eeg[f"conn_wpli_mean_{b}"])[0, 1],
                clustering=np.corrcoef(eeg[f"conn_wpli_clustering_{b}"], eeg[f"conn_wpli_mean_{b}"])[0, 1],
                mean_wpli=eeg[f"conn_wpli_mean_{b}"].median(), noise_floor=floor_w[b][off].mean())
        for b in BANDS}
print("\nwPLI graphs: correlation of each graph measure with the plain mean wPLI, across subjects;")
print("median mean wPLI, and the same quantity for unrelated noise")
print(pd.DataFrame(rows).T.round(3))

# How much does using 300 s instead of the cache's first 80/100 windows change
# band power? Same definition, different stretch of recording.
if CACHE_DIR.exists():
    cached = np.stack([np.load(CACHE_DIR / f"{sid}.npz")["bp"].mean(0).ravel() for sid in eeg.index])
    here = eeg.filter(regex=r"^bp_").to_numpy()
    r = [np.corrcoef(cached[:, j], here[:, j])[0, 1] for j in range(here.shape[1])]
    print(f"\nbp_ (first 300 s) vs subject_cache (all cached windows), across subjects: "
          f"median r = {np.median(r):.3f}, lowest r = {np.min(r):.3f}")

88 subjects x 417 EEG features:  {'existing band power': 95, 'spectral': 172, 'nonlinear': 80, 'connectivity': 70}
missing values: {'spec_paf': 8}
alpha centre of gravity: median 8.67 Hz, range 7.84-10.10 Hz

zero crossings / (2 x Hjorth mobility): median 0.96, r = 0.987   (Rice: 1 and 1 for a Gaussian signal)



wPLI graphs: correlation of each graph measure with the plain mean wPLI, across subjects;
median mean wPLI, and the same quantity for unrelated noise
       efficiency  clustering  mean_wpli  noise_floor
delta       0.990      -0.467      0.143        0.131
theta       0.904      -0.278      0.149        0.126
alpha       0.997      -0.179      0.180        0.132
beta        1.000      -0.379      0.144        0.130
gamma       0.999      -0.232      0.149        0.129



bp_ (first 300 s) vs subject_cache (all cached windows), across subjects: median r = 0.979, lowest r = 0.920


---
## Reading the output

- **Cell 6** must pass all 27 checks. Each compares a feature with a value known
  from theory, so a pass means the formula is right, not only that it runs.
- **Cell 7**: a `max |diff|` around 1e-7 means the `bp_` columns are exactly the
  cache's band power. Experiment 1 here is therefore the existing feature set,
  on a fairer 300 s.
- **Cell 9** says what the features are worth before any model sees them:
  - Zero-crossing rate is about 2 × Hjorth mobility (r ≈ 0.99), as Rice's
    formula predicts, so it should not count as a separate feature.
  - Global efficiency follows mean wPLI almost exactly (r 0.90–1.00): on a
    dense weighted graph it is the mean written another way. Clustering does
    not (r −0.2 to −0.5), because it works on weights scaled by their maximum,
    so it describes the pattern of coupling rather than its level.
  - Mean wPLI is 0.14–0.18, against a noise floor of about 0.13. Lagged coupling
    is there but small, so connectivity is unlikely to be the strongest group.
  - `bp_` over the first 300 s correlates 0.92–0.99 with the cache's longer
    average, so restricting to 300 s changes little.
  - `spec_paf` is empty for the 8 subjects whose spectrum has no peak above the
    1/f background in 6–13 Hz. Models use `spec_iaf_cog`, which is always
    defined.
- Nothing in this notebook compares AD with FTD. That happens in
  `ADvsFTD_ds004504.ipynb`, after its feature groups are fixed.